In [1]:
import pandas as pd
import glob
import os

# Lấy cả file .csv và .xlsx trong data_raw (và thư mục con nếu có)
files = sorted(glob.glob("data_raw/**/*.csv", recursive=True) + glob.glob("data_raw/**/*.xlsx", recursive=True))
print(f"Tổng số file: {len(files)}")

summary = []
for f in files:
    df = pd.read_csv(f) if f.endswith(".csv") else pd.read_excel(f)
    n_users = df['user_id'].nunique() if 'user_id' in df.columns else 'N/A'
    summary.append({
        "Tên file": os.path.basename(f),
        "Số dòng (rows)": len(df),
        "Số cột (cols)": len(df.columns),
        "Số user_id duy nhất": n_users
    })

pd.DataFrame(summary)

Tổng số file: 10


,Tên file,Số dòng (rows),Số cột (cols),Số user_id duy nhất
0,275_student_ids.csv,275,1,275
1,5.1_dialogue data.xlsx,3542,7,275
2,5.2_system_interaction_trace_data_1.xlsx,1035347,7,39
3,5.2_system_interaction_trace_data_2.xlsx,1036441,7,32
4,5.3_writing_log.xlsx,496733,4,275
5,5.4_proposal_and_score.xlsx,275,3,275
6,5.5_pre-task_biographic_survey.xlsx,276,8,267
7,5.6_pre-prio_knowledge_survey.xlsx,275,6,259
8,5.7_post-experience_using_platform_survey.xlsx,180,11,176
9,5.8_annotation_log_cleaned.xlsx,2124,7,147


In [1]:
import pandas as pd
import glob
import os

files = sorted(
    glob.glob("data_raw/**/*.csv", recursive=True) +
    glob.glob("data_raw/**/*.xlsx", recursive=True)
)

overview_rows = []
dict_rows = []
codebook_candidates = []

for idx, f in enumerate(files, start=1):
    fname = os.path.basename(f)
    print(f"[{idx}/{len(files)}] Đang đọc: {fname} ...")
    
    df = pd.read_csv(f) if f.endswith(".csv") else pd.read_excel(f)
    total_rows = len(df)
    total_cols = len(df.columns)
    n_users = df['user_id'].nunique() if 'user_id' in df.columns else 'N/A'
    key_col = 'user_id' if 'user_id' in df.columns else ''
    
    overview_rows.append({
        "STT": idx, "Tên file": fname, "Phase": "", "Đơn vị": "",
        "Số dòng": total_rows, "Số cột": total_cols,
        "Số user_id": n_users, "Key": key_col, "Mô tả": ""
    })
    
    for col in df.columns:
        dtype = str(df[col].dtype)
        n_miss = int(df[col].isna().sum())
        pct_miss = round(n_miss / total_rows * 100, 2) if total_rows > 0 else 0
        sample_vals = df[col].dropna().unique()[:3]
        sample_str = " | ".join([str(v)[:60].replace("\n", " ") for v in sample_vals])
        
        dict_rows.append({
            "Tên file": fname, "Tên cột": col, "Kiểu": dtype,
            "Ý nghĩa": "", "Định dạng": "", "Giá trị mẫu": sample_str,
            "Missing": f"{n_miss} ({pct_miss}%)", "Ghi chú": "", "Liên kết": ""
        })
        
        n_uniq = df[col].nunique(dropna=True)
        if 1 < n_uniq <= 25 and col != 'user_id':
            for val in df[col].dropna().unique():
                codebook_candidates.append({
                    "Tên file": fname, "Tên cột": col, "Mã nhãn": str(val),
                    "Gốc": "", "Tiếng Việt": "", "Ghi chú": ""
                })

# Xuất CSV không có header (header=False) để đổ thẳng từ dòng 2
pd.DataFrame(overview_rows).to_csv("tab1_tong_quan.csv", index=False, header=False, encoding="utf-8-sig")
pd.DataFrame(dict_rows).to_csv("tab2_data_dictionary.csv", index=False, header=False, encoding="utf-8-sig")
pd.DataFrame(codebook_candidates).to_csv("tab3_codebook.csv", index=False, header=False, encoding="utf-8-sig")

print("XONG! Đã tạo 3 file CSV sẵn sàng nhập vào Google Sheets.")

[1/10] Đang đọc: 275_student_ids.csv ...
[2/10] Đang đọc: 5.1_dialogue data.xlsx ...
[3/10] Đang đọc: 5.2_system_interaction_trace_data_1.xlsx ...
[4/10] Đang đọc: 5.2_system_interaction_trace_data_2.xlsx ...
[5/10] Đang đọc: 5.3_writing_log.xlsx ...
[6/10] Đang đọc: 5.4_proposal_and_score.xlsx ...
[7/10] Đang đọc: 5.5_pre-task_biographic_survey.xlsx ...
[8/10] Đang đọc: 5.6_pre-prio_knowledge_survey.xlsx ...
[9/10] Đang đọc: 5.7_post-experience_using_platform_survey.xlsx ...
[10/10] Đang đọc: 5.8_annotation_log_cleaned.xlsx ...
XONG! Đã tạo 3 file CSV sẵn sàng nhập vào Google Sheets.
